In [2]:
import dash
from dash import dcc, html, Input, Output
import plotly.express as px
import pandas as pd

# Load your data
df = pd.read_csv('depi_ungrouped_withfeatures.csv') 
df['Created at'] = pd.to_datetime(df['Created at'])

# Calculate discount percentage
df['discount_percentage'] = (df['Discount Amount'] / (df['Subtotal'] + df['Discount Amount'])) * 100

# Extract season if not already in data
if 'season' not in df.columns:
    df['season'] = df['Created at'].dt.month.map({
        1: 'Winter', 2: 'Winter', 3: 'Spring',
        4: 'Spring', 5: 'Spring', 6: 'Summer',
        7: 'Summer', 8: 'Summer', 9: 'Fall',
        10: 'Fall', 11: 'Fall', 12: 'Winter'
    })

app = dash.Dash(__name__)
server = app.server

app.layout = html.Div([
    html.H1("Seasonal Sales Analytics", style={
        'color': 'white', 
        'textAlign': 'center',
        'backgroundColor': '#1E1E1E',
        'padding': '20px'
    }),
    
    # Season Selector
    html.Div([
        dcc.Dropdown(
            id='season-selector',
            options=[{'label': 'All Seasons', 'value': 'All'}] + 
                   [{'label': season, 'value': season} 
                    for season in sorted(df['season'].unique())],
            value='All',
            style={'width': '300px', 'margin': '20px auto'}
        )
    ], style={'textAlign': 'center'}),
    
    # Key Metrics Row
    html.Div([
        html.Div([
            html.H3("Total Revenue", style={'color': '#4CAF50'}),
            html.H2(id='total-revenue', style={'margin': '10px 0'})
        ], className='metric-box'),
        
        html.Div([
            html.H3("Avg. Discount", style={'color': '#9C27B0'}),
            html.H2(id='avg-discount', style={'margin': '10px 0'})
        ], className='metric-box'),
        
        html.Div([
            html.H3("Top Product", style={'color': '#FF9800'}),
            html.H2(id='top-product', style={'margin': '10px 0'})
        ], className='metric-box')
    ], style={'display': 'flex', 'justifyContent': 'center', 'gap': '20px', 'padding': '20px'}),
    
    # First Row of Visualizations
    html.Div([
        html.Div([
            dcc.Graph(id='seasonal-sales')
        ], style={'width': '50%', 'display': 'inline-block'}),
        
        html.Div([
            dcc.Graph(id='discount-impact')
        ], style={'width': '50%', 'display': 'inline-block'})
    ]),
    
    # Second Row of Visualizations
    html.Div([
        html.Div([
            dcc.Graph(id='top-products')
        ], style={'width': '50%', 'display': 'inline-block'}),
        
        html.Div([
            dcc.Graph(id='monthly-trend')
        ], style={'width': '50%', 'display': 'inline-block'})
    ])
], style={
    'backgroundColor': '#1E1E1E',
    'color': 'white',
    'fontFamily': 'Arial, sans-serif'
})

# CSS for metric boxes
app.css.append_css({
    'external_url': '''
    .metric-box {
        background: #2E2E2E;
        padding: 20px;
        border-radius: 8px;
        text-align: center;
        min-width: 200px;
        box-shadow: 0 4px 8px rgba(0,0,0,0.2);
    }
    '''
})

@app.callback(
    [Output('seasonal-sales', 'figure'),
     Output('discount-impact', 'figure'),
     Output('top-products', 'figure'),
     Output('monthly-trend', 'figure'),
     Output('total-revenue', 'children'),
     Output('avg-discount', 'children'),
     Output('top-product', 'children')],
    [Input('season-selector', 'value')]
)
def update_dashboard(selected_season):
    filtered_df = df if selected_season == 'All' else df[df['season'] == selected_season]
    
    # 1. Seasonal Sales Breakdown
    seasonal_data = filtered_df.groupby('season')['Total'].sum().reset_index()
    season_fig = px.bar(
        seasonal_data,
        x='season',
        y='Total',
        title=f'Revenue by Season ({selected_season if selected_season != "All" else "All Seasons"})',
        color='season',
        color_discrete_sequence=px.colors.qualitative.Pastel,
        labels={'Total': 'Revenue ($)'}
    )
    
    # 2. Discount Impact Analysis (NEW)
    discount_fig = px.scatter(
        filtered_df,
        x='discount_percentage',
        y='Total',
        title='Discount % vs. Order Value',
        trendline='lowess',
        color='season',
        labels={
            'discount_percentage': 'Discount Percentage (%)',
            'Total': 'Order Total ($)'
        },
        hover_data=['Lineitem name']
    )
    
    # 3. Top Products (NEW)
    top_products = filtered_df.groupby('Lineitem name')['Lineitem quantity'].sum().nlargest(10).reset_index()
    products_fig = px.bar(
        top_products,
        x='Lineitem name',
        y='Lineitem quantity',
        title='Top 10 Ordered Items',
        color='Lineitem name',
        labels={'Lineitem quantity': 'Units Sold'}
    )
    
    # 4. Monthly Trend
    monthly_data = filtered_df.resample('M', on='Created at')['Total'].sum().reset_index()
    trend_fig = px.line(
        monthly_data,
        x='Created at',
        y='Total',
        title='Monthly Sales Trend',
        labels={'Total': 'Revenue ($)'}
    )
    
    # Calculate KPIs
    total_revenue = filtered_df['Total'].sum()
    avg_discount = filtered_df['discount_percentage'].mean()
    top_product = filtered_df.groupby('Lineitem name')['Lineitem quantity'].sum().idxmax()
    
    # Apply consistent styling
    for fig in [season_fig, discount_fig, products_fig, trend_fig]:
        fig.update_layout(
            plot_bgcolor='#2E2E2E',
            paper_bgcolor='#2E2E2E',
            font_color='white',
            hoverlabel=dict(bgcolor='#333333')
        )
    
    return (
        season_fig,
        discount_fig,
        products_fig,
        trend_fig,
        f"${total_revenue:,.0f}",
        f"{avg_discount:.1f}%",
        top_product
    )

if __name__ == '__main__':
    app.run(debug=True, port=8053)

c:\Users\Ibrah\AppData\Local\Programs\Python\Python311\Lib\site-packages\dash\resources.py:96: UserWarning:

You have set your config to `serve_locally=True` but A local version of 
    .metric-box {
        background: #2E2E2E;
        padding: 20px;
        border-radius: 8px;
        text-align: center;
        min-width: 200px;
        box-shadow: 0 4px 8px rgba(0,0,0,0.2);
    }
     is not available.
If you added this file with `app.scripts.append_script` or `app.css.append_css`, use `external_scripts` or `external_stylesheets` instead.
See https://dash.plotly.com/external-resources



C:\Users\Ibrah\AppData\Local\Temp\ipykernel_11324\2471095911.py:156: FutureWarning:

'M' is deprecated and will be removed in a future version, please use 'ME' instead.

